In [ ]:
# =============================================================
# NOTEBOOK 00 — DATA ENGINEERING
# En MLOps, este es el primer paso del pipeline:
# tomar datos crudos y dejarlos limpios y confiables.
# Un modelo entrenado con datos sucios = modelo malo en producción.
# =============================================================

# -------------------------------------------------------------
# Step 1: Import necessary libraries
# -------------------------------------------------------------

import pandas as pd          # pandas: librería principal para manipular tablas de datos (DataFrames)
import matplotlib.pyplot as plt  # matplotlib: librería base para crear gráficos y visualizaciones
import seaborn as sns        # seaborn: construida sobre matplotlib, hace gráficos estadísticos más bonitos

sns.set(style="whitegrid")   # Configura el estilo visual de los gráficos (fondo blanco con grilla)
print("✅ Libraries imported.")  # Confirmación de que las librerías se cargaron sin errores


In [ ]:
# -------------------------------------------------------------
# Step 2: Load the raw dataset
# -------------------------------------------------------------
# En MLOps, los datos crudos (raw) se guardan en data/raw/ y NUNCA se modifican.
# Siempre trabajamos con copias para mantener reproducibilidad.

print("📥 Loading dataset...")  # Mensaje informativo para saber en qué paso estamos

df = pd.read_csv("../data/raw/house_data.csv")  # Lee el CSV y lo convierte en un DataFrame (tabla en memoria)
                                                  # '../' sube un nivel desde notebooks/ hacia la raíz del proyecto

print("✅ Dataset loaded successfully!")          # Confirma que el archivo se leyó sin errores
print("Initial data shape:", df.shape)            # .shape devuelve (filas, columnas) — útil para saber el tamaño del dataset
print("\n🔍 Preview of data:")                    # '\n' agrega una línea en blanco para mejor legibilidad
print(df.head())                                   # .head() muestra las primeras 5 filas — primer vistazo a los datos


In [ ]:
# -------------------------------------------------------------
# Step 3: Understand structure — schema, datatypes, nulls
# -------------------------------------------------------------
# En MLOps, entender el 'schema' (estructura) de los datos es crítico.
# Si el schema cambia en producción (ej: una columna desaparece), el pipeline falla.

print("\n📊 Data types and null value check:")  # Encabezado informativo
print(df.info())                                  # .info() muestra: nombre de columnas, tipo de dato (int, float, object)
                                                  # y cuántos valores NO nulos hay por columna

print("\n❓ Any missing values?")               # Encabezado para la siguiente verificación
print(df.isnull().sum())                          # .isnull() devuelve True/False por cada celda
                                                  # .sum() cuenta cuántos True hay por columna = total de valores faltantes


In [ ]:
# Step 3b: Drop malformed rows (like the one with missing columns)
# En MLOps, los datos incompletos o inválidos deben eliminarse ANTES de entrenar.
# Entrenar con NaN o valores absurdos corrompe el modelo.

df = df.dropna()                    # Elimina TODAS las filas que tengan al menos un valor nulo (NaN)
                                    # Reasignamos a 'df' para sobreescribir el DataFrame con la versión limpia

df = df[df['price'] > 10000]        # Filtro de sanidad: descarta casas con precio menor a $10,000
                                    # Una casa con precio $0 o $500 es claramente un error de datos

df = df[df['sqft'] > 200]           # Filtro de sanidad: descarta casas con menos de 200 pies cuadrados
                                    # Un valor tan bajo probablemente es un error de ingreso de datos


In [ ]:
# -------------------------------------------------------------
# Step 4: Standardize column names (basic cleanup)
# -------------------------------------------------------------
# En MLOps, los nombres de columnas deben ser consistentes entre entornos.
# Si en producción llegan columnas con espacios o mayúsculas distintas, el pipeline falla.

print("\n🧹 Cleaning column names (lowercase, underscores)...")  # Mensaje informativo

df.columns = df.columns.str.strip()          # .str.strip() elimina espacios al inicio y al final de cada nombre de columna
df.columns = df.columns.str.lower()          # .str.lower() convierte todo a minúsculas (ej: 'Price' → 'price')
df.columns = df.columns.str.replace(" ", "_")  # .str.replace() reemplaza espacios por guiones bajos (ej: 'year built' → 'year_built')
                                               # Esto se puede encadenar en una sola línea (como estaba antes), aquí separado para claridad

print("✅ Column names standardized:")        # Confirmación
print(df.columns.tolist())                    # .tolist() convierte el índice de columnas a una lista Python normal para verla fácil


In [ ]:
# -------------------------------------------------------------
# Step 5: Basic stats check (spot outliers or issues)
# -------------------------------------------------------------
# En MLOps, los outliers (valores extremos) pueden degradar el rendimiento del modelo.
# .describe() es el primer paso para detectarlos visualmente.

print("\n📈 Basic summary stats for numeric columns:")  # Encabezado
print(df.describe())  # .describe() calcula para cada columna numérica:
                      # count (total de valores), mean (promedio), std (desviación estándar),
                      # min, 25%, 50% (mediana), 75%, max
                      # Si el max es 1000x el promedio, hay un outlier


In [ ]:
# -------------------------------------------------------------
# Step 6: Visual check for distributions (e.g. price, sqft)
# -------------------------------------------------------------
# En MLOps, visualizar la distribución de datos ayuda a entender si el modelo
# va a ver datos 'normales' o muy sesgados (skewed) en producción.

print("\n📊 Plotting distribution of house prices...")  # Mensaje informativo

plt.figure(figsize=(8, 5))              # Crea una figura nueva de 8 pulgadas de ancho x 5 de alto
sns.histplot(df["price"], kde=True, bins=10)  # histplot: dibuja un histograma de la columna 'price'
                                              # kde=True: agrega una curva de densidad suavizada encima
                                              # bins=10: divide el rango de precios en 10 barras
plt.title("Distribution of House Prices")    # Título del gráfico
plt.xlabel("Price")                           # Etiqueta del eje X
plt.ylabel("Count")                           # Etiqueta del eje Y (cantidad de casas por rango de precio)
plt.show()                                    # Renderiza y muestra el gráfico en pantalla


In [ ]:
# -------------------------------------------------------------
# Step 7: Check for obvious data issues
# -------------------------------------------------------------
# En MLOps, validar que no existan valores negativos en columnas que no deberían tenerlos
# es parte del 'data validation' — un paso crítico antes de cualquier entrenamiento.

print("\n🔍 Looking for negative or unrealistic values...")  # Encabezado

for col in ['price', 'sqft', 'bedrooms', 'bathrooms', 'year_built']:  # Itera sobre las columnas que deben ser positivas
    invalid = df[df[col] < 0]          # Filtra filas donde el valor de la columna es negativo
                                        # df[condición] devuelve solo las filas donde la condición es True
    if not invalid.empty:              # .empty devuelve True si el DataFrame no tiene filas
                                        # 'not invalid.empty' = hay filas con valores negativos
        print(f"⚠️ Found negative values in {col}:\n{invalid}")  # f-string: inserta variables dentro del string
    else:
        print(f"✅ No negative values found in {col}.")  # Todo OK para esta columna


In [ ]:
# -------------------------------------------------------------
# Step 8: Drop duplicates if any
# -------------------------------------------------------------
# En MLOps, los duplicados en el dataset de entrenamiento causan 'data leakage':
# el modelo aprende la misma fila varias veces y se sobreajusta (overfitting).

print("\n🧹 Checking for duplicate rows...")  # Encabezado

duplicates = df.duplicated().sum()  # .duplicated() devuelve True para cada fila que es copia de una anterior
                                     # .sum() cuenta cuántas filas duplicadas hay en total

if duplicates > 0:                   # Si hay al menos un duplicado...
    print(f"⚠️ Found {duplicates} duplicate rows. Dropping them...")  # Avisa cuántos hay
    df = df.drop_duplicates()        # .drop_duplicates() elimina todas las filas duplicadas, conservando la primera ocurrencia
else:
    print("✅ No duplicate rows found.")  # No hay duplicados, el dataset está limpio


In [ ]:
# -------------------------------------------------------------
# Step 9: Save the cleaned version (optional here)
# -------------------------------------------------------------
# En MLOps, guardar los datos limpios en data/interim/ o data/processed/ es una buena práctica.
# Permite que el siguiente paso del pipeline (EDA, Feature Engineering) use datos ya validados
# sin tener que repetir toda la limpieza desde cero.

print("\n💾 Saving cleaned version to interim file (optional step)...")  # Encabezado

# Descomenta la siguiente línea si quieres guardar el DataFrame limpio en disco:
# df.to_csv("../data/interim/cleaned_house_data_v1.csv", index=False)
# .to_csv(): guarda el DataFrame como archivo CSV
# index=False: no guarda el índice numérico de pandas como columna extra en el CSV

print("✅ Data engineering preprocessing complete!")  # Fin del notebook — datos listos para EDA
